# 02 - Raw Data Audit

This notebook audits the raw datasets considered for the project **Cameroonian International Migration Trends (2015-2024)**.

The objective is to identify which sources are reliable enough to support the final analytical questions and which sources should remain contextual only.

The audit focuses on:

- file structure and available sheets
- country and year coverage
- missing values, duplicated rows and special codes
- whether Cameroon is directly identifiable
- whether the source can support analytical tables without mixing incompatible indicators


In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path("..") # The notebook is located in the "notebooks" folder
DATA_RAW = PROJECT_ROOT / "data" / "raw"

GLOBAL_PATH = DATA_RAW / "global"
EUROPE_PATH = DATA_RAW / "europe"
CANADA_PATH = DATA_RAW / "canada"
USA_PATH = DATA_RAW / "usa"
JAPAN_PATH = DATA_RAW / "japan"
UNHCR_PATH = DATA_RAW / "unhcr"
CONTEXT_PATH = DATA_RAW / "context"

In [ ]:
def audit_dataset(
    df,
    dataset_name,
    year_col=None,
    country_cols=None,
    focus_value="Cameroon",
    max_display=10
):
    print("=" * 70)
    print(f"AUDIT : {dataset_name}")
    print("=" * 70)

    # 1. Dataset size
    print("\n[1] Dataset size")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    # 2. Columns
    print("\n[2] Columns")
    print(df.columns.tolist())

    # 3. Data types
    print("\n[3] Types")
    print(df.dtypes)

    # 4. Missing values
    print("\n[4] Missing values")
    print(df.isna().sum())

    # 5. Duplicate rows
    print("\n[5] Duplicates")
    print(df.duplicated().sum())

    # 6. Preview
    print("\n[6] Preview")
    display(df.head())

    # 7. Available years
    if year_col and year_col in df.columns:
        print("\n[7] Available years")
        years = sorted(df[year_col].dropna().unique())
        print(years)

    # 8. Available countries
    if country_cols:
        print("\n[8] Available countries")
        for col in country_cols:
            if col in df.columns:
                print(f"\nColumn: {col}")
                countries = df[col].dropna().unique()
                print(f"Number of distinct values: {len(countries)}")
                print(f"Examples: {list(countries[:max_display])}")

    # 9. Presence of Cameroon
    if country_cols:
        print("\n[9] Presence of Cameroon")
        for col in country_cols:
            if col in df.columns:
                has_cameroon = df[col].astype(str).str.contains(focus_value, case=False, na=False).any()
                print(f"{col} : {has_cameroon}")

## Audit - UN DESA

Objective: verify the structure of the main Q1 dataset on global destination countries for migrants originating from Cameroon.


In [ ]:
undesa = pd.read_csv(GLOBAL_PATH / "undesa_cameroon_global_destinations.csv")
audit_dataset(undesa, "UN DESA")

In [ ]:
print("Available years:", sorted(undesa["Year"].unique()))
print("Available origins:", undesa["Origin"].nunique())
print("Available destinations:", undesa["Destination"].nunique())

In [ ]:
undesa[undesa["Origin"] == "Cameroon"].head()

### Findings

- Main dataset for Q1: it contains origin country, destination country, year and migrant stock.
- Long structure: each row represents one observation.
- Available years: 1990, 1995, 2000, 2005, 2010, 2015, 2020 and 2024. The dataset provides selected reference years rather than a complete annual series.
- Regional aggregates require caution: in addition to countries, some rows correspond to groups such as `World`, `Africa`, `Asia`, `Eastern Africa` or `Western Africa`.
- Numeric columns may need type conversion before analysis.


## Audit - Eurostat First Permits

Objective: verify entry reason data for Cameroonian citizens in Europe.


In [ ]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx")
xls.sheet_names

In [ ]:
sheet1 = pd.read_excel(EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx", sheet_name="Feuille 1", header = None)
audit_dataset(sheet1, "Eurostat first permits - sheet 1")

### Findings

- Several sheets are available.
- Each sheet corresponds to one entry reason type.
- The actual data starts after metadata rows.
- The file must be transformed from wide format to long format.
- Eurostat flags may be present and should be handled carefully.


## Audit - Eurostat Long-Term Residents

Objective: verify long-term resident data for Cameroonian citizens in Europe.


In [ ]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_long_term_residents_cameroon.xlsx")
xls.sheet_names

In [ ]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_long_term_residents_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

In [ ]:
audit_dataset(
    df=sheet1_raw,
    dataset_name="Eurostat long-term residents - Sheet 1 (raw)",
    year_col=None,
    country_cols=None
)

### Findings

- The workbook contains a `Summary` sheet and three data sheets.
- The actual data does not start on the first row.
- Sheet 1 appears to correspond to the total legal framework.
- Visible years range from 2015 to 2024.
- Nationality is already filtered to `Cameroon`.
- Eurostat flags may be present.


## Audit - Eurostat Status Changes

Objective: verify immigration status change data for Cameroonian citizens in Europe.


In [ ]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_status_changes_cameroon.xlsx")
xls.sheet_names

In [ ]:
audit_dataset(
    df=sheet1_raw,
    dataset_name="Eurostat status changes - Sheet 1 (raw)",
    year_col=None,
    country_cols=None
)

### Findings

- The workbook contains a `Summary` sheet and several data sheets.
- The actual data does not start on the first row.
- The dataset mainly covers 2020 to 2024.
- It supports Q3.
- It helps observe administrative status changes after arrival.
- The cleaning should start with the simplest sheet, likely total age and total sex.


## Audit - Eurostat Citizenship Acquisition

Objective: verify citizenship acquisition data for Cameroonian citizens in Europe.


In [ ]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_citizenship_acquisition_cameroon.xlsx")
xls.sheet_names

In [ ]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_citizenship_acquisition_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

sheet1_raw.head(15)

In [ ]:
audit_dataset(
    df=sheet1_raw,
    dataset_name="Eurostat citizenship acquisition - Sheet 1 (raw)",
    year_col=None,
    country_cols=None
)

### Findings

- The workbook contains a `Summary` sheet and many data sheets.
- The actual data does not start on the first row.
- The dataset covers 2015 to 2024.
- It supports Q3.
- It helps observe naturalization and citizenship acquisition.
- The cleaning should start with the simplest sheet: total age and total sex.


## Audit - Statistics Canada Immigration Status

Objective: verify the contextual Statistics Canada file on immigration status for people linked to Cameroon in Canada.


In [ ]:
CANADA_PATH = DATA_RAW / "canada"

In [ ]:
statcan_raw = pd.read_csv(
    CANADA_PATH / "statcan_cameroon_immigrant_status.csv",
    header=None,
    skiprows=8
)

statcan_raw.head(15)

In [ ]:
audit_dataset(
    df=statcan_raw,
    dataset_name="StatCan - raw immigrant status",
    year_col=None,
    country_cols=None
)

### Audit Conclusion

- Useful dataset for Canada context.
- Not a priority source for annual trend analysis.
- Cleaning is required because metadata appears at the top of the file.


## Audit - IRCC Permanent Residents

Objective: verify the dataset on Cameroonian permanent residents in Canada.


In [ ]:
xls = pd.ExcelFile(CANADA_PATH / "ircc_permanent_residents_cameroon.xlsx")
xls.sheet_names

In [ ]:
ircc_pr_raw = pd.read_excel(
    CANADA_PATH / "ircc_permanent_residents_cameroon.xlsx",
    sheet_name=0,
    header=None
)

ircc_pr_raw.head(15)

In [ ]:
audit_dataset(
    df=ircc_pr_raw,
    dataset_name="IRCC - Permanent residents (raw)",
    year_col=None,
    country_cols=None
)

In [ ]:
ircc_pr_raw[ircc_pr_raw.astype(str).apply(lambda col: col.str.contains("Cameroon", case=False, na=False)).any(axis=1)]

### Findings

- Excel file with multi-line headers.
- Wide structure.
- Coverage from 2015 to 2026.
- Includes the row `Cameroon, Federal Republic of`.
- Special values such as `--` may be present.
- Main Canada-specific dataset.


## Audit - IRCC Permanent Residents by Category

Objective: verify the dataset on Cameroonian permanent residents by immigration category.


In [ ]:
xls = pd.ExcelFile(CANADA_PATH / "ircc_permanent_residents_cameroon_by_category.xlsx")
xls.sheet_names

In [ ]:
ircc_pr_cat_raw = pd.read_excel(
    CANADA_PATH / "ircc_permanent_residents_cameroon_by_category.xlsx",
    sheet_name=0,
    header=None
)

ircc_pr_cat_raw.head(20)

In [ ]:
audit_dataset(
    df=ircc_pr_cat_raw,
    dataset_name="IRCC - Permanent residents by category (raw)",
    year_col=None,
    country_cols=None
)

In [ ]:
ircc_pr_cat_raw[
    ircc_pr_cat_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

### Findings

- Excel file with multi-line headers.
- Wide structure.
- Coverage from 2015 to 2026.
- Includes a `Cameroon, Federal Republic of - Total` block.
- Immigration subcategories appear below the total.
- Special values such as `--` may be present.
- Useful for a Canada-specific extension of Q2 and Q3.

### Audit Conclusion

- Main Canada-specific dataset.
- Useful for admission categories.
- Cleaning is required: rebuild the header, extract the Cameroon block, handle categories and reshape to long format.


## Audit - IRCC Study Permits

Objective: verify the dataset on study permits issued to Cameroonian citizens in Canada.


In [ ]:
xls = pd.ExcelFile(CANADA_PATH / "ircc_study_permits_cameroon.xlsx")
xls.sheet_names

In [ ]:
ircc_study_raw = pd.read_excel(
    CANADA_PATH / "ircc_study_permits_cameroon.xlsx",
    sheet_name=0,
    header=None
)

ircc_study_raw.head(20)

In [ ]:
audit_dataset(
    df=ircc_study_raw,
    dataset_name="IRCC - Study permits (raw)",
    year_col=None,
    country_cols=None
)

In [ ]:
ircc_study_raw[
    ircc_study_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

### Findings

- Excel file with multi-line headers.
- Wide structure.
- Coverage from 2015 to 2026.
- Includes the row `Cameroon, Federal Republic of`.
- Special values such as `--` may be present.
- Useful for a Canada-specific extension of Q2.

### Audit Conclusion

- Main Canada-specific dataset.
- Useful for analyzing study permits.
- Cleaning is required: rebuild the header, reshape to long format and handle `--`.


## Audit - IRCC Temporary-to-Permanent Transitions

Objective: verify the dataset on transitions from temporary status to permanent residence in Canada.


In [ ]:
xls = pd.ExcelFile(CANADA_PATH / "ircc_temp_to_pr_cameroon.xlsx")
xls.sheet_names

In [ ]:
ircc_temp_pr_raw = pd.read_excel(
    CANADA_PATH / "ircc_temp_to_pr_cameroon.xlsx",
    sheet_name=0,
    header=None
)

ircc_temp_pr_raw.head(20)

In [ ]:
audit_dataset(
    df=ircc_temp_pr_raw,
    dataset_name="IRCC - Temporary to permanent (raw)",
    year_col=None,
    country_cols=None
)

In [ ]:
ircc_temp_pr_raw[
    ircc_temp_pr_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

### Findings

- Excel file with multi-line headers.
- Wide structure.
- Coverage from 2015 to 2026.
- The file appears to be organized by province or territory and category.
- The presence of `Cameroon` must be verified.
- If `Cameroon` is absent, this file should be treated as Canada context rather than a direct Cameroon dataset.

### Audit Conclusion

- Interesting dataset for understanding temporary-to-permanent transition logic in Canada.
- Keep it only if useful as context.
- If `Cameroon` is absent, do not treat it as a main Cameroon dataset.


## Audit - USA Lawful Permanent Residents

Objective: verify the DHS dataset on lawful permanent residents linked to Cameroon.


In [ ]:
USA_PATH = DATA_RAW / "usa"

In [ ]:
xls = pd.ExcelFile(USA_PATH / "usa_lawful_permanent_residents_cameroon.xlsx")
xls.sheet_names

In [ ]:
usa_lpr_raw = pd.read_excel(
    USA_PATH / "usa_lawful_permanent_residents_cameroon.xlsx",
    sheet_name=0,
    header=None
)

usa_lpr_raw.head(20)

In [ ]:
audit_dataset(
    df=usa_lpr_raw,
    dataset_name="USA DHS - Lawful permanent residents (raw)",
    year_col=None,
    country_cols=None
)

In [ ]:
usa_lpr_raw[
    usa_lpr_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

### Findings

- Excel file with several tables or sheets.
- DHS data is often organized by table.
- Rows for `Cameroon` must be verified depending on the sheet.
- Coverage is useful up to 2022.
- DHS years are fiscal years and must be documented as such.
- Useful for a USA-specific Q3 extension.

### Audit Conclusion

- Useful dataset for the United States.
- It does not fully cover the 2015-2024 period.
- Fiscal years require caution.
- Cleaning is required: identify the correct table, extract the `Cameroon` row and harmonize years.


## Audit - USA Refugee Arrivals

Objective: verify the DHS / OHSS dataset on refugee arrivals from Cameroon to the United States.


In [ ]:
xls = pd.ExcelFile(USA_PATH / "usa_refugee_arrivals_cameroon.xlsx")
xls.sheet_names

In [ ]:
usa_refugees_raw = pd.read_excel(
    USA_PATH / "usa_refugee_arrivals_cameroon.xlsx",
    sheet_name=0,
    header=None
)

usa_refugees_raw.head(20)

In [ ]:
audit_dataset(
    df=usa_refugees_raw,
    dataset_name="USA DHS - Refugee arrivals (raw)",
    year_col=None,
    country_cols=None
)

In [ ]:
usa_refugees_raw[
    usa_refugees_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

### Findings

- DHS / OHSS Excel file.
- Tables may be organized by year or series.
- The `Cameroon` row must be located in the useful table.
- Coverage ranges from 2015 to 2024.
- DHS years are fiscal years and must be documented as such.
- Useful for a USA-specific refugee or asylum angle.

### Audit Conclusion

- Useful dataset for a USA extension.
- Specific angle: refugees.
- Do not treat this dataset as a measure of all Cameroonian migration to the USA.
- Cleaning is required: identify the correct table, extract `Cameroon` and harmonize fiscal years.


## Audit - USA ACS Selected Population

Objective: verify the ACS descriptive profile file for the Cameroon-related population in the United States.


In [ ]:
xls = pd.ExcelFile(USA_PATH / "usa_acs_selected_population_cameroon.xlsx")
xls.sheet_names

In [ ]:
usa_acs_raw = pd.read_excel(
    USA_PATH / "usa_acs_selected_population_cameroon.xlsx",
    sheet_name=0,
    header=None
)

usa_acs_raw.head(20)

In [ ]:
audit_dataset(
    df=usa_acs_raw,
    dataset_name="USA ACS - Selected population (raw)",
    year_col=None,
    country_cols=None
)

In [ ]:
usa_acs_raw[
    usa_acs_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

### Findings

- ACS / Census file.
- Likely includes a data sheet with a descriptive profile.
- Useful as a contextual point, not as a long annual series.
- May include margins of error.
- Secondary contextual dataset rather than a central analytical source.

### Audit Conclusion

- Useful dataset for USA descriptive context.
- Not a main dataset for 2015-2024 trend analysis.
- Keep it as complementary context.
- Cleaning is required: identify the useful table and retain estimates and margins of error when relevant.


## Audit - Japan Visas Related to Cameroon

Objective: verify the file on Japan visas related to Cameroonian citizens.


In [ ]:
JAPAN_PATH = DATA_RAW / "japan"

In [ ]:
japan_visas_raw = pd.read_csv(
    JAPAN_PATH / "japan_visas_cameroon.csv"
)

japan_visas_raw.head(20)

In [ ]:
audit_dataset(
    df=japan_visas_raw,
    dataset_name="Japan visas Cameroon",
    year_col=None,
    country_cols=None
)

In [ ]:
japan_visas_raw.columns.tolist()

In [ ]:
japan_visas_raw[
    japan_visas_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
].head(20)

### Findings

- CSV file, simpler than the Canada and USA Excel files.
- Potentially useful for an Asia-focused complement.
- Likely has more limited period coverage than the Europe and Canada datasets.
- More useful as international context than as a central analytical source.
